# Task 3: Supervised Classification / Regression Modeling & Tuning

## Objective
Train and compare at least four distinct classification architectures, tune hyperparameters with stratified cross-validation, evaluate using precision, recall, F1, ROC-AUC and a confusion matrix, select a champion model using validation metrics, and serialize the trained champion model.

**Dataset:** Titanic tabular dataset from OpenML. The previous task page did not provide an official dataset file, so the same public dataset is reused for continuity.

**Important:** Model selection is based on cross-validation on the training set. The final holdout test set is used only for final evaluation, reducing test-set selection bias.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay, roc_curve
)
import joblib

RANDOM_STATE = 42

## 1. Load and prepare the data

In [ ]:
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame.copy()

target = 'survived'
drop_cols = ['survived', 'boat', 'body', 'home.dest', 'name', 'ticket']
model_df = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

X = model_df.drop(columns=[target])
y = model_df[target].astype(int)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print('Dataset shape:', df.shape)
print('Numeric:', numeric_features)
print('Categorical:', categorical_features)
display(df.head())

## 2. Train/test split

The test set remains untouched during hyperparameter tuning. All preprocessing is fitted inside each model pipeline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 3. Reusable preprocessing pipeline

Numerical features are median-imputed and scaled. Categorical features are imputed and one-hot encoded. The transformations are learned only from training folds during cross-validation.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

## 4. Define four model architectures and search spaces

In [ ]:
models = {
    'Logistic Regression': (
        LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
        {
            'model__C': [0.1, 1, 10],
            'model__solver': ['liblinear']
        }
    ),
    'Random Forest': (
        RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
        {
            'model__n_estimators': [200, 400],
            'model__max_depth': [None, 5, 10],
            'model__min_samples_split': [2, 5]
        }
    ),
    'HistGradientBoosting': (
        HistGradientBoostingClassifier(random_state=RANDOM_STATE),
        {
            'model__learning_rate': [0.05, 0.1],
            'model__max_iter': [100, 200],
            'model__max_leaf_nodes': [15, 31]
        }
    ),
    'SVM': (
        SVC(probability=True, random_state=RANDOM_STATE),
        {
            'model__C': [0.5, 1, 10],
            'model__kernel': ['rbf', 'linear'],
            'model__gamma': ['scale']
        }
    )
}


## 5. Hyperparameter tuning with stratified 5-fold cross-validation

ROC-AUC is used as the primary validation metric for model selection. Precision, recall and F1 are also recorded.

In [ ]:
searches = {}
cv_rows = []

for name, (estimator, param_grid) in models.items():
    pipe = Pipeline([
        ('preprocess', preprocessor),
        ('model', estimator)
    ])

    search = GridSearchCV(
        pipe,
        param_grid=param_grid,
        scoring='roc_auc',
        cv=cv,
        n_jobs=-1,
        refit=True,
        return_train_score=False
    )
    search.fit(X_train, y_train)
    searches[name] = search

    cv_rows.append({
        'model': name,
        'cv_roc_auc': search.best_score_,
        'best_params': search.best_params_
    })

cv_results = pd.DataFrame(cv_rows).sort_values('cv_roc_auc', ascending=False)
display(cv_results)

## 6. Holdout test evaluation

Evaluate the tuned version of every architecture on the untouched test set.

In [ ]:
evaluation_rows = []
test_predictions = {}

for name, search in searches.items():
    model = search.best_estimator_
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)[:, 1]
    test_predictions[name] = (pred, proba)

    evaluation_rows.append({
        'model': name,
        'precision': precision_score(y_test, pred, zero_division=0),
        'recall': recall_score(y_test, pred, zero_division=0),
        'f1': f1_score(y_test, pred, zero_division=0),
        'roc_auc': roc_auc_score(y_test, proba)
    })

evaluation = pd.DataFrame(evaluation_rows).sort_values('roc_auc', ascending=False)
display(evaluation)

## 7. Confusion matrix for each model

In [ ]:
for name, (pred, proba) in test_predictions.items():
    cm = confusion_matrix(y_test, pred)
    print(name)
    print(cm)
    print()


## 8. ROC-AUC curves

In [ ]:
plt.figure(figsize=(9, 7))
for name, (pred, proba) in test_predictions.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], linestyle='--', label='Random baseline')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-AUC Comparison')
plt.legend()
plt.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Champion model selection

The champion is selected from the tuned models using the highest mean cross-validated ROC-AUC on the training data. The holdout test set is not used to choose the champion.

In [ ]:
champion_name = cv_results.iloc[0]['model']
champion_search = searches[champion_name]
champion_model = champion_search.best_estimator_

print('Champion model:', champion_name)
print('Best CV ROC-AUC:', champion_search.best_score_)
print('Best parameters:', champion_search.best_params_)

champion_pred = champion_model.predict(X_test)
champion_proba = champion_model.predict_proba(X_test)[:, 1]
print('Holdout precision:', precision_score(y_test, champion_pred, zero_division=0))
print('Holdout recall:', recall_score(y_test, champion_pred, zero_division=0))
print('Holdout F1:', f1_score(y_test, champion_pred, zero_division=0))
print('Holdout ROC-AUC:', roc_auc_score(y_test, champion_proba))

## 10. Serialize the champion model

The complete preprocessing + model pipeline is saved as one `.joblib` file so the same transformations are applied at inference time.

In [ ]:
joblib.dump(champion_model, 'champion_model.joblib')
print('Saved: champion_model.joblib')

## Final checklist

- [x] Four distinct model architectures
- [x] Stratified K-fold cross-validation
- [x] GridSearchCV hyperparameter optimization
- [x] Precision, recall and F1 evaluation
- [x] ROC-AUC and ROC curves
- [x] Confusion matrices
- [x] Champion selected using validation ROC-AUC
- [x] Complete champion pipeline serialized with joblib
- [x] Test set kept separate from model selection
